# Oil–Water Barrier: Governing Math

The **oil–water barrier** models phase-separated edge transport: plasma density $n_p$ ("oil") and lithium vapor $n_v$ ("water") coupled at a moving interface, with tritium $n_T$ bred in the vapor blanket.

## Steady-state coupled system

$$D_p \frac{d^2 n_p}{dx^2} = \alpha\, n_p n_v$$

$$D_v \frac{d^2 n_v}{dx^2} - v_v \frac{dn_v}{dx} = -\alpha\, n_p n_v$$

Tritium obeys an advection–diffusion equation with Li breeding source $\sigma_{Li}\,\Phi_n\, n_v$.

## Peclet extraction criterion

$$\mathrm{Pe} = \frac{v_v \,\delta}{D_T}, \quad \delta = \left(\frac{D_p D_v}{\alpha^2 n_0 n_{wall}}\right)^{1/4}$$

When $\mathrm{Pe} > 1$, outward vapor sweep dominates diffusion — tritium is extracted at the wall.

---

## Full six-field oil–water system (VISION §3)

fuselk's **novel contribution** is a coupled multi-physics edge model — not a single diffusion equation. In steady state along the radial coordinate $x$ (core → wall):

| Field | Governing equation |
|-------|-------------------|
| Plasma density $n_p$ | $\displaystyle\frac{d}{dx}\!\left(D_p \frac{dn_p}{dx}\right) - \alpha\, n_p n_v - \sigma_{strip}\, n_p \Phi_{ext} = 0$ |
| Vapor density $n_v$ | $\displaystyle\frac{d}{dx}\!\left(D_v \frac{dn_v}{dx}\right) - v_v \frac{dn_v}{dx} + \alpha\, n_p n_v - S_{breed}(x) = 0$ |
| Tritium $n_T$ | $\displaystyle\frac{d}{dx}\!\left(-D_T \frac{dn_T}{dx} + v_v n_T\right) = \sigma_{Li}\, \Phi_n\, n_v(x) + S_\mu(x)$ |
| Muon density $n_\mu$ | $\displaystyle\frac{d}{dx}\!\left(D_\mu \frac{dn_\mu}{dx}\right) - v_\mu \frac{dn_\mu}{dx} + R_{strip}(x) - \lambda_\mu n_\mu = 0$ |
| Plasma energy $T_p$ | $\displaystyle\frac{d}{dx}\!\left(\kappa_p \frac{dT_p}{dx}\right) - P_{brem} - \beta\,\alpha\, n_p n_v T_p - P_\mu = 0$ |
| Vapor energy $T_v$ | $\displaystyle\frac{d}{dx}\!\left(\kappa_v \frac{dT_v}{dx}\right) - v_v \rho_v C_p \frac{dT_v}{dx} + \beta\,\alpha\, n_p n_v T_p - P_{rad} + Q_{laser} = 0$ |

**Interface thickness** (mutual conversion scale):
$$\delta = \left(\frac{D_p D_v}{\alpha^2 n_0 n_w}\right)^{1/4}$$

The reduced solver in `physics/pde_solver.py` implements the $(n_p, n_v)$ subsystem + linear tritium sub-solve; energy residuals live in `physics/energy_balance.py`.

### Proof sketch: Peclet extraction (Pe > 1 ⇒ outward tritium flux)

Consider tritium on $x \in [0,L]$ with $n_T(0)$ fixed and wall sink at $x=L$. Steady advection–diffusion:
$$-D_T n_T'' + v_v n_T' = S(x), \quad S \ge 0$$

Multiply by integrating factor and integrate. When $v_v \delta / D_T > 1$, boundary-layer theory places the tritium peak away from the core; the wall flux
$$\Gamma_T = -D_T n_T'(L) + v_v n_T(L)$$
is **advection-dominated** at the wall. Hence $\mathrm{Pe} = v_v \delta / D_T > 1$ is the **necessary scale condition** for outward fuel extraction rather than core reflux — the key fuselk fuel-cycle closure criterion that OpenMC/IMAS do not model.


In [ ]:
import sys
from pathlib import Path

from deepiri_fuselk.data.notebook_loaders import resolve_repo_root

repo = resolve_repo_root()

try:
    import deepiri_fuselk  # noqa: F401
except ImportError:
    sys.path.insert(0, str(repo / "src"))
    import deepiri_fuselk  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
%config InlineBackend.figure_formats = ["retina"]

from deepiri_fuselk.data.imas_loader import load_imas_hdf5
from deepiri_fuselk.data.notebook_loaders import (
    ensure_fetched_data,
    imas_to_synthetic_shot,
    list_shots,
    load_corpus_arrays,
    load_odl_meta,
    manifest_summary,
    resolve_data_root,
)

data_root = ensure_fetched_data(resolve_data_root(repo), n_shots=100, max_odl=50)
cmod_shots = list_shots(data_root, source="cmod")
syn_shots = list_shots(data_root, source="synthetic")
corpus = load_corpus_arrays(data_root)
manifest = manifest_summary(data_root)

print(f"Data lake: {data_root}")
print(f"  C-Mod (ODL) shots: {len(cmod_shots)}")
print(f"  Synthetic shots:   {len(syn_shots)}")
print(f"  ELM corpus frames: {len(corpus['labels'])} (elm_rate={corpus['elm_rate']:.2f})")
for row in manifest:
    print(f"  [{row['source']}] {row['status']} — {row['shots']} shots @ {row['fetched_at'][:19]}")


In [ ]:
from deepiri_fuselk.barrier.breeding_blanket import evaluate_breeding_blanket, tritium_breeding_ratio
from deepiri_fuselk.physics import (
    PDEParameters,
    interface_thickness,
    peclet_criterion,
    solve_oil_water_steady,
    solve_oil_water_transient,
)

params = PDEParameters()
delta = interface_thickness(params)
pe = peclet_criterion(params)

print(f"Interface thickness δ = {delta:.4f}")
print(f"Peclet number Pe = {pe:.3f}  ({'outward sweep wins' if pe > 1 else 'diffusion dominates'})")

result = solve_oil_water_steady(n_grid=128)
state = result.state
tbr = tritium_breeding_ratio(state, params)
blanket = evaluate_breeding_blanket(state, params)

print(f"Newton converged: {result.converged}, residual = {result.residual:.2e}")
print(f"TBR = {tbr:.3f}, extraction efficiency = {blanket.extraction_efficiency:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(state.x, state.n_p, label=r"$n_p$ plasma", color="C0")
axes[0].plot(state.x, state.n_v, label=r"$n_v$ vapor", color="C1")
axes[0].set_xlabel("x"); axes[0].set_ylabel("density"); axes[0].legend(); axes[0].set_title("Steady profiles")

axes[1].plot(state.x, state.n_T, color="C2")
axes[1].set_xlabel("x"); axes[1].set_ylabel(r"$n_T$")
axes[1].set_title("Tritium distribution")

axes[2].plot(state.x, state.T_p, label=r"$T_p$")
axes[2].plot(state.x, state.T_v, label=r"$T_v$")
axes[2].set_xlabel("x"); axes[2].legend(); axes[2].set_title("Temperature proxies")

plt.tight_layout()
plt.show()


## Experiment: Peclet sweep vs. breeding ratio

In [ ]:
vapor_speeds = np.linspace(0.1, 1.2, 15)
pe_values, tbr_values = [], []

for v_v in vapor_speeds:
    p = PDEParameters(v_v=v_v)
    res = solve_oil_water_steady(n_grid=64, params=p)
    pe_values.append(peclet_criterion(p))
    tbr_values.append(tritium_breeding_ratio(res.state, p))

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(pe_values, tbr_values, "o-", color="C3", label="TBR")
ax1.axvline(1.0, color="gray", ls="--", label="Pe = 1")
ax1.set_xlabel("Peclet number Pe"); ax1.set_ylabel("Tritium breeding ratio")
ax1.set_title("Fuel-cycle sensitivity to vapor sweep")
ax1.legend()
plt.tight_layout()
plt.show()


## Energy-balance residuals (full six-field system)

In [ ]:
\
from deepiri_fuselk.physics.energy_balance import plasma_energy_residual, vapor_energy_residual

res_p = plasma_energy_residual(state, params)
res_v = vapor_energy_residual(state, params)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(state.x, res_p, label=r"plasma energy residual $\mathcal{R}_p$")
ax.plot(state.x, res_v, label=r"vapor energy residual $\mathcal{R}_v$")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("x"); ax.legend(); ax.set_title("Energy equation residuals on steady PDE state")
plt.tight_layout()
plt.show()


## Transient evolution

In [ ]:
history = solve_oil_water_transient(n_grid=64, t_end=0.5, dt=0.005)
times = np.arange(len(history)) * 0.005

fig, ax = plt.subplots(figsize=(8, 4))
for i in [0, len(history) // 4, len(history) // 2, -1]:
    ax.plot(history[i].x, history[i].n_v, label=f"t = {times[i]:.2f}")
ax.set_xlabel("x"); ax.set_ylabel(r"$n_v$")
ax.set_title("Vapor density transient")
ax.legend()
plt.tight_layout()
plt.show()


## Real C-Mod density drives PDE boundary conditions

In [ ]:
# Scale oil–water PDE from fetched ne profiles across the ODL shot corpus
from deepiri_fuselk.barrier.breeding_blanket import tritium_breeding_ratio

discharge_pe, discharge_tbr, odl_rates = [], [], []
for path in cmod_shots[:12]:
    shot = load_imas_hdf5(path)
    meta = load_odl_meta(path)
    ne_edge = float(np.mean(shot.ne_profile.values[-3:]))
    p = PDEParameters(n0=ne_edge * 1e-6, v_v=0.55)
    res = solve_oil_water_steady(n_grid=64, params=p)
    discharge_pe.append(peclet_criterion(p))
    discharge_tbr.append(tritium_breeding_ratio(res.state, p))
    odl_rates.append(meta["elm_event_rate"] if meta else 0.0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(odl_rates, discharge_tbr, c=discharge_pe, cmap="plasma", s=60)
axes[0].set_xlabel("ODL density-limit phase rate"); axes[0].set_ylabel("TBR")
axes[0].set_title("Fuel cycle vs. experimental precursor labels")

axes[1].hist(discharge_pe, bins=8, color="C0", alpha=0.8, label="Pe from C-Mod ne")
axes[1].axvline(1.0, color="k", ls="--", label="Pe = 1 extraction threshold")
axes[1].set_xlabel("Peclet number"); axes[1].legend()
axes[1].set_title("Peclet distribution on real discharges")

plt.tight_layout()
plt.show()
